# **TFM Project - Machine Learning for Drug Discovery in Neurodegenerative Diseases**
# **[Part 1] Download Raw Bioactivity Data**

Carla D. Di Monno

In **Part 1**, data from the ChEMBL database is collected, filtered and saved.
---

## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is an open-access repository containing curated bioactivity information on approximately 2.9 million compounds. Its current dataset is derived from over 99,000 scientific papers and 24.2 million activity measurements, covering a range of 17,803 therapeutic targets and more than 2,200 cell lines. [Data updated as of 28 July 2025; ChEMBL version 36].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [1]:
!pip install chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00


## **Importing libraries**

In [33]:
# Import necessary libraries
import pandas as pd
import re
from chembl_webresource_client.new_client import new_client

In [3]:
# Initialise the API resources
target = new_client.target
activity = new_client.activity
molecule = new_client.molecule

## **Search and filter for therapeutic target**

In [15]:
def get_and_save_bioactivity_data(target_name, filename):
    """
    Searches for a target in ChEMBL, filters for human and single protein,
    retrieves IC50 bioactivity data (binding/functional assays),
    and saves the results to a CSV file.

    Args:
        target_name (str): The name of the target to search for.
        filename (str): The name of the CSV file to save the data.

    Returns:
        pandas.DataFrame: The DataFrame containing the bioactivity data.
    """
    print(f"\n--- Processing target: {target_name} ---")

    # 1. Target search
    target_query = new_client.target.search(target_name)
    targets = pd.DataFrame.from_dict(target_query)

    # Select and retrieve bioactivity data for 'Homo sapiens' and 'SINGLE PROTEIN'
    selected_targets = targets.loc[(targets.organism == 'Homo sapiens') &
                                   (targets.target_type == 'SINGLE PROTEIN')]

    if selected_targets.empty:
        print(f"No 'Homo sapiens' SINGLE PROTEIN found for {target_name}.")
        return pd.DataFrame()

    selected_target_chembl_id = selected_targets['target_chembl_id'].values[0]
    print(f"Target ID for {target_name}: {selected_target_chembl_id}")

    # 2. Retrieve bioactivity data
    res = new_client.activity.filter(target_chembl_id=selected_target_chembl_id) \
                                  .filter(standard_type="IC50") \
                                  .filter(assay_type__in=['B', 'F'])

    df = pd.DataFrame.from_dict(res)

    print(f"Shape of raw data for {target_name}: {df.shape}")

    # 3. Save to CSV
    df.to_csv(filename, index=False)
    print(f"Bioactivity data for {target_name} saved to {filename}")

    return df

## **Apply to all therapeutic targets**

### **1- Target search for Acetylcholinesterase**

In [16]:
# Acetylcholinesterase
ache_df = get_and_save_bioactivity_data('acetylcholinesterase', 'AChE_bioactivity_raw_data.csv')


--- Processing target: acetylcholinesterase ---
Target ID for acetylcholinesterase: CHEMBL220
Shape of raw data for acetylcholinesterase: (9558, 46)
Bioactivity data for acetylcholinesterase saved to AChE_bioactivity_raw_data.csv


### **2- Target search for Monoamino oxidase B**

In [17]:
# Monoamino oxidase B
maob_df = get_and_save_bioactivity_data('monoamino oxidase B', 'MAO-B_bioactivity_raw_data.csv')


--- Processing target: monoamino oxidase B ---
Target ID for monoamino oxidase B: CHEMBL2039
Shape of raw data for monoamino oxidase B: (8599, 46)
Bioactivity data for monoamino oxidase B saved to MAO-B_bioactivity_raw_data.csv


### **3- Target search for Leucine-rich repeat kinase 2**

In [28]:
# Leucine-rich repeat kinase 2 (LRRK2)
lrrk2_df = get_and_save_bioactivity_data('leucine-rich repeat kinase 2', 'LRRK2_bioactivity_raw_data.csv')


--- Processing target: leucine-rich repeat kinase 2 ---
Target ID for leucine-rich repeat kinase 2: CHEMBL1075104
Shape of raw data for leucine-rich repeat kinase 2: (6601, 46)
Bioactivity data for leucine-rich repeat kinase 2 saved to LRRK2_bioactivity_raw_data.csv


### 3.1- Special conditions for LRRK2: Check for mutations

In [37]:
# Classify LRRK2 bioactivity data as either WT or G2019S mutant based on the
# assay description and variant sequence information

# Define Inclusion Keywords
inclusion_pattern = r'G2019S|G2019-S|MUTANT|MUTATION'

# Define Exclusion Keywords (to avoid false positives)
exclusion_pattern = r'SELECTIVITY OVER|VERSUS|VS|COMPARED TO|WT OVER'

def label_lrrk2_variant(row):
    desc = str(row['assay_description']).upper()

    # Checks for an explicit 'variant_sequence' field
    if 'variant_sequence' in row and pd.notnull(row['variant_sequence']):
        return 'G2019S'

    # Searches the 'assay_description' for inclusion keywords
    if re.search(inclusion_pattern, desc):
        if re.search(exclusion_pattern, desc):
            return 'WT'
        return 'G2019S'

    return 'WT'

lrrk2_df['variant_type'] = lrrk2_df.apply(label_lrrk2_variant, axis=1)

# Final Statistics
print(f"Total LRRK2 records: {len(lrrk2_df)}")
print(f"G2019S records: {len(lrrk2_df[lrrk2_df['variant_type'] == 'G2019S'])}")
print(f"Wild Type records: {len(lrrk2_df[lrrk2_df['variant_type'] == 'WT'])}")
print(f"New actual percentage of mutants: {(len(lrrk2_df[lrrk2_df['variant_type'] == 'G2019S'])/len(lrrk2_df))*100:.2f}%")

# Split and save the dataframes
lrrk2_g2019s_df = lrrk2_df[lrrk2_df['variant_type'] == 'G2019S'].copy()
lrrk2_wt_df = lrrk2_df[lrrk2_df['variant_type'] == 'WT'].copy()

lrrk2_g2019s_df.to_csv('LRRK2_G2019S_bioactivity_raw_data.csv', index=False)
print("LRRK2 G2019S bioactivity data saved to LRRK2_G2019S...csv")

lrrk2_wt_df.to_csv('LRRK2_WT_bioactivity_raw_data.csv', index=False)
print("LRRK2 Wild Type bioactivity data saved to LRRK2_WT...csv")

Total LRRK2 records: 6601
G2019S records: 3261
Wild Type records: 3340
New actual percentage of mutants: 49.40%
LRRK2 G2019S bioactivity data saved to LRRK2_G2019S...csv
LRRK2 Wild Type bioactivity data saved to LRRK2_WT...csv


In [41]:
# Take a random sample of those who claim to be mutants
check_sample = lrrk2_df[lrrk2_df['assay_description'].str.contains('G2019S', case=False, na=False)].sample(5)

for i, row in check_sample.iterrows():
    variant_seq_value = row.get('variant_sequence', 'N/A') # Safely get variant_sequence, default to 'N/A'
    print(f"ID: {row['molecule_chembl_id']} | Variant Type: {row['variant_type']} | Variant Sequence (if available): {variant_seq_value} | Description: {row['assay_description']}\n")

ID: CHEMBL5802583 | Variant Type: G2019S | Variant Sequence (if available): N/A | Description: Biochemical Assay: Materials:LRRK2 G2019S enzymeSubstrate (LRRKtide)ATPTR-FRET dilution bufferpLRRKtide antibody384-well assay plateDMSOEnzyme Reaction Conditions50 mM Tris pH 7.5, 10 mM MgCl2, 1 mM EGTA, 0.01% Brij-35.2 mM DTT5 nM LRRK2134 μM ATP60 minute reaction time23° C. reaction temperature10 μL total reaction volumeDetection Reaction Conditions1× TR-FRET dilution buffer10 mM EDTA2 nM antibody23° C. reaction temperature10 μL total reaction volumeCompounds were prepared by initially diluting to 1 mM with DMSO. 35 μL of reference compound solution, 35 μL of test compound solution, and 35 μL HPE were successively added to the source plate (384-well assay plate, Labcyte). The plates were centrifuged at 2500 rpm for 1 minute and sealed in foil. POD was used to perform a 3.162 fold serial dilution and 100 nL of reference compound solution, test compound solution, HPE and ZPE were transferred 

### **4- Target search for Glycogen synthase kinase-3 beta**

In [19]:
# Glycogen synthase kinase-3 beta (GSK3B)
gsk3b_df = get_and_save_bioactivity_data('glycogen synthase kinase-3 beta', 'GSK3B_bioactivity_raw_data.csv')


--- Processing target: glycogen synthase kinase-3 beta ---
Target ID for glycogen synthase kinase-3 beta: CHEMBL262
Shape of raw data for glycogen synthase kinase-3 beta: (4944, 46)
Bioactivity data for glycogen synthase kinase-3 beta saved to GSK3B_bioactivity_raw_data.csv
